In [1]:
!pip install -U bitsandbytes>=0.46.1
# temp fix

zsh:1: 0.46.1 not found


In [ ]:
import os
import sys
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Configurazione del percorso e Cache (Colab o Locale)
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    
    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    
    # Impostiamo HF_HOME prima di importare hf_hub o transformers
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    print(f"Ambiente Colab e Google Drive configurati. Cache modelli in: {hf_cache_dir}")
except ImportError:
    print("Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.")

/home/zizzis/Code/Politecnico/Large_Language_Models/CLARITY/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ambiente locale rilevato. Verrà usata la cache Hugging Face di default.


In [ ]:
# 2. Gestione del Token di Autenticazione
hf_token = None
try:
    from google.colab import userdata
    print("Acquisizione token da Google Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Lettura token dal file .env locale...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("Errore: File .env non trovato.")

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Autenticazione Hugging Face completata.")
else:
    print("Attenzione: Token HF non trovato. Possibili errori nel download del modello.")

Lettura token dal file .env locale...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Autenticazione Hugging Face completata.


## Direct clarity classification (Classificazione diretta della chiarezza)

Questo task consiste nel prevedere direttamente una delle tre macro-categorie di chiarezza (il livello più alto della tassonomia), valutando il numero di interpretazioni che la risposta ammette. Le etichette sono:
   * **Clear Reply** (Risposta chiara): ammette una sola interpretazione.
   * **Ambivalent Reply** (Risposta ambivalente): viene fornita una risposta valida ma strutturata in modo da prestarsi a molteplici interpretazioni.
   * **Clear Non-Reply** (Non-risposta chiara): il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

In [ ]:
# 3. Caricamento del Modello (Quantizzazione 4-bit)
model_id = "meta-llama/Llama-3.1-8B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama 3.1 non ha un pad token di default, usiamo l'eos_token per il fine-tuning
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Inizializzazione configurazione BitsAndBytes per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Trasferimento dei pesi del modello in VRAM...")

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16  # Corretto da dtype
)

print("Caricamento completato con successo. Sistema pronto per l'inferenza.")

Inizializzazione configurazione BitsAndBytes per meta-llama/Llama-3.1-8B-Instruct...
Trasferimento dei pesi del modello in VRAM...


Fetching 4 files:   0%|          | 0/4 [04:18<?, ?it/s]


In [ ]:
# 4. Preparazione per PEFT (LoRA / DoRA)
print("Preparazione del modello per il training a 4-bit...")
model = prepare_model_for_kbit_training(model)

# Configurazione DoRA basata sui layer di attenzione di Llama
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True # Attiviamo DoRA come suggerito per performance migliori
)

print("Applicazione degli adattatori LoRA/DoRA...")
model = get_peft_model(model, peft_config)

# Stampa un riepilogo per verificare di star addestrando solo una frazione dei parametri
model.print_trainable_parameters()

In [ ]:
# 5. Caricamento e Formattazione del Dataset Reale (Task 1: Clarity Classification)
import os
from datasets import load_from_disk

# Percorso della cartella che contiene i file .arrow, dataset_info.json, ecc.
# Assicurati che questo path sia corretto rispetto a dove hai estratto i dati
dataset_path = "dataset/QEvasion/train" 

print(f"Caricamento del dataset da: {dataset_path}...")
try:
    # Carica il dataset salvato in formato arrow/directory di Hugging Face
    dataset = load_from_disk(dataset_path)
    print(f"Dataset caricato con successo! Numero di esempi: {len(dataset)}")
except Exception as e:
    print(f"Errore nel caricamento del dataset. Verifica il percorso. Dettaglio: {e}")

# Definiamo il System Prompt specifico per il Task 1
system_prompt_task1 = """Sei un esperto analista di discorsi politici. Il tuo compito è classificare la risposta di un politico a una domanda diretta in una delle seguenti 3 categorie:
1. 'Clear Reply': La risposta ammette una sola interpretazione chiara.
2. 'Ambivalent Reply': Viene fornita una risposta, ma è strutturata in modo da prestarsi a molteplici interpretazioni o essere vaga.
3. 'Clear Non-Reply': Il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

Rispondi SOLO con il nome esatto della categoria (Clear Reply, Ambivalent Reply o Clear Non-Reply), senza aggiungere altro testo."""

def format_prompt_task1(example):
    """
    Formatta ogni riga del dataset nel formato chat richiesto da Llama 3.1,
    utilizzando le colonne reali del dataset QEvasion.
    """
    # Usiamo le chiavi estratte dal tuo file dataset_info.json
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    label = example.get('clarity_label', '')
    
    user_message = f"Domanda: {domanda}\nRisposta del politico: {risposta}"
    
    # Strutturiamo i messaggi
    messages = [
        {"role": "system", "content": system_prompt_task1},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": str(label)}
    ]
    
    # Applichiamo il template nativo del tokenizer di Llama
    testo_formattato = tokenizer.apply_chat_template(messages, tokenize=False)
    
    return {"text": testo_formattato}

print("Formattazione del dataset in corso...")
# Applichiamo la funzione a tutto il dataset
# Rimuoviamo le vecchie colonne per alleggerire la memoria, tenendo solo la colonna 'text' utile al training
formatted_dataset = dataset.map(
    format_prompt_task1, 
    remove_columns=dataset.column_names 
)

# Stampiamo un esempio per verificare che il template sia applicato correttamente
print("\n--- Esempio di Prompt Formattato per Llama 3.1 ---")
print(formatted_dataset[0]["text"])

In [ ]:
# 6. TRAINING COMPLETO (Senza limiti di step)
import torch
import os
from transformers import TrainingArguments
from trl import SFTTrainer

output_model_dir = "./risultati_clarity_task1"
os.makedirs(output_model_dir, exist_ok=True)

training_arguments = TrainingArguments(
    output_dir=output_model_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_strategy="epoch",            # Salva un checkpoint alla fine di ogni epoca
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,                        # Sfrutta la tua GPU potente (A100/L4)
    max_grad_norm=0.3,
    num_train_epochs=3,               # Training completo su 3 epoche (consigliato)
    warmup_steps=100,                        
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none"                  # Evita di chiedere login a WandB
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    args=training_arguments,
    peft_config=None,                 # Il modello è già PeftModel
    processing_class=tokenizer
)

print("Inizio dell'addestramento completo! 🚀")
trainer.train()

# Salvataggio definitivo
final_save_path = f"{output_model_dir}/modello_finale"
trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"Modello salvato con successo in: {final_save_path}")

In [ ]:
# 7. Test e Inferenza sul Dataset di Test
import torch
import os
from datasets import load_from_disk
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("--- FASE DI TEST ---")

# 1. Caricamento o verifica del modello
# Se stiamo eseguendo questa cella in un momento diverso e il modello non è in memoria, lo ricarichiamo
if 'model' not in locals():
    print("Modello non trovato in memoria. Inizializzo il caricamento dal Drive/Locale...")
    
    # Percorsi
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    lora_path = "./risultati_clarity_task1/modello_finale"
    
    tokenizer = AutoTokenizer.from_pretrained(lora_path) # Carichiamo il tokenizer salvato
    
    # Ri-configuriamo la quantizzazione
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    # Carichiamo il modello base a 4-bit
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16
    )
    
    # Fondiamo il modello base con i nostri adattatori addestrati
    print(f"Caricamento adattatori LoRA da: {lora_path}")
    model = PeftModel.from_pretrained(base_model, lora_path)
    print("Modello caricato con successo!")
else:
    print("Modello già presente in memoria. Procedo con l'inferenza usando il modello corrente.")

model.eval() # Impostiamo il modello in modalità valutazione

# 2. Caricamento del dataset di Test
test_dataset_path = "dataset/QEvasion/test"
print(f"\nCaricamento dataset di test da: {test_dataset_path}...")
try:
    test_dataset = load_from_disk(test_dataset_path)
    print(f"Dataset di test caricato! Numero totale di esempi: {len(test_dataset)}")
except Exception as e:
    print(f"Errore nel caricamento del dataset di test: {e}")

# Ri-definiamo il system prompt (se non fosse in memoria)
system_prompt_task1 = """Sei un esperto analista di discorsi politici. Il tuo compito è classificare la risposta di un politico a una domanda diretta in una delle seguenti 3 categorie:
1. 'Clear Reply': La risposta ammette una sola interpretazione chiara.
2. 'Ambivalent Reply': Viene fornita una risposta, ma è strutturata in modo da prestarsi a molteplici interpretazioni o essere vaga.
3. 'Clear Non-Reply': Il rispondente rifiuta apertamente o dichiara di non poter fornire le informazioni richieste.

Rispondi SOLO con il nome esatto della categoria (Clear Reply, Ambivalent Reply o Clear Non-Reply), senza aggiungere altro testo."""

# 3. Funzione di Inferenza
def predici_classe(example):
    domanda = example.get('interview_question', '')
    risposta = example.get('interview_answer', '')
    
    messages = [
        {"role": "system", "content": system_prompt_task1},
        {"role": "user", "content": f"Domanda: {domanda}\nRisposta del politico: {risposta}"}
    ]
    
    # add_generation_prompt=True aggiunge <|start_header_id|>assistant<|end_header_id|> per innescare la generazione
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Tokenizziamo
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generazione
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,      # Pochi token: vogliamo solo il nome dell'etichetta
            temperature=0.0,        # Temperature a 0 per avere l'output più probabile e deterministico (zero creatività)
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decodifichiamo SOLO i token generati dal modello (ignoriamo il prompt)
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    predizione = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    return predizione

# 4. Esecuzione del test sui primi 5 esempi
print("\n--- INIZIO VALUTAZIONE SUI PRIMI 5 ESEMPI ---")
num_test = min(5, len(test_dataset))

esatti = 0
for i in range(num_test):
    esempio = test_dataset[i]
    vero_label = str(esempio.get('clarity_label', '')).strip()
    
    predizione = predici_classe(esempio)
    
    corretto = "✅" if predizione.lower() == vero_label.lower() else "❌"
    if predizione.lower() == vero_label.lower():
        esatti += 1
        
    print(f"\nEsempio {i+1}:")
    print(f"Vera Etichetta  : {vero_label}")
    print(f"Predizione LLM  : {predizione} {corretto}")

print(f"\nAccuratezza sui primi {num_test} esempi: {esatti}/{num_test} ({(esatti/num_test)*100:.1f}%)")

In [ ]:
import torch
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Preparazione liste per il confronto
y_true = []
y_pred = []

print(f"Inizio valutazione su tutto il dataset di test ({len(test_dataset)} esempi)...")

# 2. Ciclo di inferenza
for i in tqdm(range(len(test_dataset))):
    esempio = test_dataset[i]
    
    # Etichetta reale
    vero_label = str(esempio.get('clarity_label', '')).strip()
    y_true.append(vero_label)
    
    # Predizione del modello
    try:
        predizione = predici_classe(esempio)
        y_pred.append(predizione)
    except Exception as e:
        print(f"Errore all'esempio {i}: {e}")
        y_pred.append("Error")

# 3. Generazione del report
print("\n" + "="*20 + " CLASSIFICATION REPORT " + "="*20)
# Specifichiamo le etichette per assicurarci che l'ordine sia coerente
target_names = ["Clear Reply", "Ambivalent Reply", "Clear Non-Reply"]

# Nota: Se il modello predice stringhe leggermente diverse, scikit-learn le gestirà, 
# ma è meglio normalizzare se necessario.
report = classification_report(y_true, y_pred)
print(report)

# 4. Matrice di Confusione (Visualizzazione Grafica)
print("\nGenerazione Matrice di Confusione...")
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(list(set(y_true))), 
            yticklabels=sorted(list(set(y_true))))
plt.xlabel('Predizioni LLM')
plt.ylabel('Etichette Reali')
plt.title('Matrice di Confusione - Clarity Task 1')
plt.show()

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import json

# 1. Estrazione dei log dall'oggetto trainer
# trainer.state.log_history contiene una lista di dizionari con i log registrati
history = trainer.state.log_history

# 2. Trasformazione in un DataFrame di Pandas per manipolarli facilmente
df_logs = pd.DataFrame(history)

# 3. Pulizia dei dati
# I log contengono sia i dati di training che (eventualmente) quelli di eval e riepilogo finale.
# Teniamo solo le colonne interessanti e rimuoviamo le righe vuote per la loss.
columns_to_keep = ['step', 'loss', 'learning_rate', 'epoch', 'eval_loss', 'eval_accuracy']
df_filtered = df_logs[[c for c in columns_to_keep if c in df_logs.columns]].dropna(subset=['loss'], how='all')

# 4. Salvataggio su Google Drive
csv_filename = f"{output_model_dir}/training_logs.csv"
json_filename = f"{output_model_dir}/training_logs_full.json"

df_filtered.to_csv(csv_filename, index=False)

# Salviamo anche la versione JSON completa (che contiene metadati extra)
with open(json_filename, "w") as f:
    json.dump(history, f, indent=4)

print(f"Dati della loss salvati con successo!")
print(f"CSV creato: {csv_filename}")
print(f"JSON completo creato: {json_filename}")

# Visualizziamo le ultime righe del log per conferma
print("\nUltimi log registrati:")
print(df_filtered.tail())